In [24]:
import requests
from bs4 import BeautifulSoup
import json
import re

url = "https://www.topcv.vn/viec-lam/nhan-vien-van-hanh-may-tien-phay-cnc-thu-nhap-8-12-trieu-tai-ho-chi-minh/2091652.html?ta_source=BoxAttractiveJob_LinkDetail"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    "Referer": "https://www.topcv.vn/",
    "Connection": "keep-alive",
}

session = requests.Session()
session.headers.update(headers)

# Warm-up request để lấy cookie trước khi vào trang chi tiết
try:
    session.get("https://www.topcv.vn/", timeout=20)
except requests.RequestException:
    pass

try:
    resp = session.get(url, timeout=20, allow_redirects=True)
except requests.RequestException as e:
    print(f"Request failed: {e}")
    resp = None

html = ""
if resp is not None:
    if resp.status_code == 200:
        html = resp.text
    else:
        print(f"Cannot access page directly (HTTP {resp.status_code}). Continue with empty content.")

soup = BeautifulSoup(html, "html.parser")

job_data = {}

# 1) Ưu tiên parse JSON-LD (thường chứa dữ liệu chuẩn của bài tuyển dụng)
for script in soup.find_all("script", type="application/ld+json"):
    raw = script.get_text(strip=True)
    if not raw:
        continue
    try:
        data = json.loads(raw)
    except Exception:
        continue

    items = data if isinstance(data, list) else [data]
    for item in items:
        if isinstance(item, dict) and item.get("@type") == "JobPosting":
            job_data["title"] = item.get("title")
            job_data["description"] = item.get("description")
            job_data["datePosted"] = item.get("datePosted")
            job_data["validThrough"] = item.get("validThrough")

            org = item.get("hiringOrganization", {})
            if isinstance(org, dict):
                job_data["company"] = org.get("name")

            loc = item.get("jobLocation", {})
            if isinstance(loc, dict):
                addr = loc.get("address", {})
                if isinstance(addr, dict):
                    job_data["location"] = addr.get("addressLocality") or addr.get("addressRegion")

            salary = item.get("baseSalary")
            if salary:
                job_data["salary"] = salary
            break

# 2) Fallback: lấy trực tiếp từ HTML nếu thiếu
if "title" not in job_data:
    h1 = soup.find("h1")
    if h1:
        job_data["title"] = h1.get_text(strip=True)

if "company" not in job_data:
    company_tag = soup.select_one("a.company-name, .company-name, .job-detail__company a")
    if company_tag:
        job_data["company"] = company_tag.get_text(strip=True)

# In kết quả
print("=== Job Data ===")
for k, v in job_data.items():
    print(f"{k}: {v}")

=== Job Data ===
title: Nhân Viên Vận Hành Máy Tiện/ Phay CNC- Thu Nhập 8-12 Triệu- Tại Hồ Chí Minh
description: <h2>Mô tả công việc</h2>
<ul><li>Nhận bản vẽ và chương trình gia công</li><li>Chuẩn bị máy và thiết lập, vận hành máy</li><li>Phối hợp QC kiểm tra chất lượng sản phẩm sau khi chạy máy xong.</li><li>Kiểm tra kích thước sản phẩm bằng các dụng cụ đo: thước cặp, panme, đồng hồ so...</li><li>Thực hiện đúng quy trình công nghệ gia công, vệ sinh máy trước mà sau khi làm việc</li><li>Thực hiện công việc khác theo sự sắp xếp, hướng dẫn của cấp Quản lý.</li></ul>
<h2>Yêu cầu ứng viên</h2>
<ul><li><b>Nam/nữ, tuổi từ 18-40 tuổi.</b></li><li><b>Không yêu cầu kinh nghiệm, các bạn Sinh viên mới ra trường sẽ được đào tạo</b></li><li><b>Đọc hiểu bản vẽ kỹ thuật cơ khí cơ bản</b></li><li>Biết sử dụng thước cặp, panme, đồng hồ so</li><li>Tốt nghiệp Trung cấp nghề hoặc Cao đẳng ngành cơ khí, cắt gọt kim loại hoặc liên quan.</li><li>Trung thực, trách nhiệm, chịu khó ham học hỏi</li></ul>
<h2>Quy

In [14]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--window-size=1920,1080")
options.add_argument(f"--user-agent={headers['User-Agent']}")

driver = webdriver.Chrome(options=options)

try:
    driver.get(url)
    WebDriverWait(driver, 20).until(
        lambda d: d.execute_script("return document.readyState") == "complete"
    )
    html = driver.page_source
finally:
    driver.quit()

soup = BeautifulSoup(html, "html.parser")
job_data = {}

for script in soup.find_all("script", type="application/ld+json"):
    raw = script.get_text(strip=True)
    if not raw:
        continue
    try:
        json_data = json.loads(raw)
    except Exception:
        continue

    items = json_data if isinstance(json_data, list) else [json_data]
    for item in items:
        if isinstance(item, dict) and item.get("@type") == "JobPosting":
            job_data["title"] = item.get("title")
            job_data["description"] = item.get("description")
            job_data["datePosted"] = item.get("datePosted")
            job_data["validThrough"] = item.get("validThrough")

            org = item.get("hiringOrganization", {})
            if isinstance(org, dict):
                job_data["company"] = org.get("name")

            loc = item.get("jobLocation", {})
            if isinstance(loc, dict):
                addr = loc.get("address", {})
                if isinstance(addr, dict):
                    job_data["location"] = addr.get("addressLocality") or addr.get("addressRegion")

            salary = item.get("baseSalary")
            if salary:
                job_data["salary"] = salary
            break

print("=== Job Data ===")
for k, v in job_data.items():
    print(f"{k}: {v}")

=== Job Data ===
title: Fullstack Team Lead (NodeJS / ReactJS) - 5 Năm Kinh Nghiệm - Làm Việc Tại Lê Đức Thọ, HN - Phúc Lợi Tốt, Thu Nhập 25 Triệu +++
description: <h2>Mô tả công việc</h2>
<ul><li>Dẫn dắt team phát triển WebApp (NodeJS + ReactJS)</li><li>Làm việc trực tiếp với CEO/khách hàng, đảm bảo delivery</li><li>Thiết kế kiến trúc hệ thống, đưa ra giải pháp kỹ thuật</li><li>Hands-on coding, review code, xử lý các issue phức tạp</li><li>Quản lý task, phân công công việc, theo dõi tiến độ</li><li>Xây dựng quy trình (Git, CI/CD) và phát triển team</li></ul>
<h2>Yêu cầu ứng viên</h2>
<ul><li>5+ năm kinh nghiệm Fullstack, 1–2 năm làm Lead</li><li>Thành thạo NodeJS, NestJS, ReactJS, NextJS, TypeScript</li><li>Hiểu REST API, system design, database (PostgreSQL/MySQL)</li><li>Có kinh nghiệm CI/CD, Docker là lợi thế</li><li>Tư duy ownership, giao tiếp tốt, làm việc trực tiếp với khách hàng</li><li>Ưu tiên: đã làm outsource, biết tiếng Anh/Nhật, từng build system từ 0→1</li></ul>
<h2>Quyền 